<a href="https://colab.research.google.com/github/sr606/Python-Practice/blob/main/Mermaid_v9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
📁 lineage_engine/
🧠 agent.py
import os
from graphviz import Source

from parsing.parser import split_into_stages
from parsing.extractor import extract_raw_blocks
from parsing.lineage_order import order_by_lineage

from llm.llm_normalizer import normalize_stage
from llm.llm_dot_compiler import generate_dot

from rendering.dot_sanitizer import clean_and_validate_dot


INPUT_FOLDER = "data/input"
OUTPUT_FOLDER = "data/output"


def process_file(file_path: str, file_name: str):

    with open(file_path, "r", encoding="utf-8") as f:
        pseudo = f.read()

    print(f"\nProcessing: {file_name}")

    raw_stages = split_into_stages(pseudo)

    structured_stages = []

    for stage in raw_stages:

        raw = extract_raw_blocks(stage["block"])

        normalized = normalize_stage(stage["stage_id"], raw)

        normalized["inputs"] = raw.get("inputs", [])
        normalized["outputs"] = raw.get("outputs", [])

        structured_stages.append(normalized)

    # Order stages based on dataset flow
    structured_stages = order_by_lineage(structured_stages)

    workflow_name = file_name.replace(".txt", "")

    dot_code = generate_dot(workflow_name, structured_stages)

    dot_code = clean_and_validate_dot(dot_code)

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)

    output_path = os.path.join(OUTPUT_FOLDER, workflow_name)

    src = Source(dot_code)
    src.render(output_path, format="pdf", cleanup=True)

    print(f"Rendered: {output_path}.pdf")


def run_agent():

    if not os.path.exists(INPUT_FOLDER):
        raise FileNotFoundError("data/input folder not found")

    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(".txt")]

    if not files:
        print("No input files found.")
        return

    for file in files:
        process_file(os.path.join(INPUT_FOLDER, file), file)


if __name__ == "__main__":
    run_agent()
📁 config/settings.py
import os
from dotenv import load_dotenv

load_dotenv()

AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
📁 parsing/parser.py
import re


def split_into_stages(text):

    pattern = r"// --- \[(.*?)\] \[Lines.*?\] ---([\s\S]*?)(?=// --- \[|$)"
    matches = re.findall(pattern, text)

    stages = []

    for header, body in matches:

        name_match = re.search(r":\s*(.*)", header)
        if not name_match:
            continue

        stage_id = name_match.group(1).strip()

        stages.append({
            "stage_id": stage_id,
            "block": body.strip()
        })

    return stages
📁 parsing/extractor.py
import re


def extract_raw_blocks(block):

    sql_match = re.search(r"SQL:(.*?)(StageType:|Output:|$)", block, re.DOTALL)
    sql_block = sql_match.group(1) if sql_match else ""

    transform_match = re.search(r"Transformations:(.*)", block, re.DOTALL)
    transform_block = transform_match.group(1) if transform_match else ""

    input_datasets = re.findall(
        r"Input:\s*←\s*dataset_\d+\s*\(([^)]+)\)", block
    )

    output_datasets = re.findall(
        r"Output:\s*→\s*dataset_\d+\s*\(([^)]+)\)", block
    )

    return {
        "sql": sql_block.strip(),
        "transform": transform_block.strip(),
        "inputs": input_datasets,
        "outputs": output_datasets
    }
📁 parsing/lineage_order.py
def order_by_lineage(stages):

    stage_map = {s["stage_id"]: s for s in stages}
    output_to_stage = {}

    for s in stages:
        for out in s.get("outputs", []):
            output_to_stage[out] = s["stage_id"]

    ordered = []
    visited = set()

    def visit(stage):
        if stage["stage_id"] in visited:
            return

        for inp in stage.get("inputs", []):
            if inp in output_to_stage:
                parent = stage_map[output_to_stage[inp]]
                visit(parent)

        visited.add(stage["stage_id"])
        ordered.append(stage)

    for s in stages:
        visit(s)

    return ordered
📁 llm/llm_normalizer.py
import json
from openai import AzureOpenAI
from config.settings import (
    AZURE_OPENAI_API_KEY,
    AZURE_OPENAI_API_VERSION,
    AZURE_OPENAI_ENDPOINT,
    AZURE_OPENAI_DEPLOYMENT
)

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)


PROMPT = """
Extract structured lineage from SQL + Transform block.

Return STRICT JSON only:

{
  "stage_id": "",
  "primary_tables": [],
  "joins": [
    {
      "join_type": "",
      "table": "",
      "condition": ""
    }
  ],
  "filters": [],
  "transformations": [
    {
      "target": "",
      "logic": ""
    }
  ]
}

Rules:
- Extract first-level FROM as primary table
- Extract top-level joins only
- Ignore nested subqueries
- Max 3 joins
- Max 6 transformations
- Copy text exactly
"""


def normalize_stage(stage_id, raw):

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0,
        messages=[
            {"role": "system", "content": "You extract SQL lineage strictly."},
            {"role": "user", "content": PROMPT + f"\nStage ID: {stage_id}\n\nSQL:\n{raw['sql']}\n\nTRANSFORM:\n{raw['transform']}"}
        ]
    )

    content = response.choices[0].message.content.strip()

    try:
        return json.loads(content)
    except:
        return {
            "stage_id": stage_id,
            "primary_tables": [],
            "joins": [],
            "filters": [],
            "transformations": []
        }
📁 llm/llm_dot_compiler.py
import json
from openai import AzureOpenAI
from config.settings import (
    AZURE_OPENAI_API_KEY,
    AZURE_OPENAI_API_VERSION,
    AZURE_OPENAI_ENDPOINT,
    AZURE_OPENAI_DEPLOYMENT
)

client = AzureOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
)


PROMPT = """
You are a professional Graphviz workflow compiler.

Generate COMPLETE valid DOT code.

STRICT RULES:
- Use rankdir=LR
- bgcolor="white"
- Helvetica font
- HTML table labels
- subgraph cluster_workflow
- Use stage_id exactly
- Create edges from inputs/outputs
- No markdown
- Output must start with: digraph WORKFLOW_NAME {
"""


def generate_dot(workflow_name, stages):

    payload = json.dumps({
        "workflow_name": workflow_name,
        "stages": stages
    }, ensure_ascii=False)

    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT,
        temperature=0,
        messages=[
            {"role": "system", "content": "You generate strictly valid Graphviz DOT."},
            {"role": "user", "content": PROMPT + "\n\nWorkflow JSON:\n" + payload}
        ],
        stop=["```"]
    )

    return response.choices[0].message.content.strip()
📁 rendering/dot_sanitizer.py
import re
import html


def clean_and_validate_dot(dot_text):

    dot_text = dot_text.strip()

    dot_text = re.sub(r"```.*?```", "", dot_text, flags=re.DOTALL)
    dot_text = html.unescape(dot_text)

    if not dot_text.startswith("digraph"):
        raise ValueError("DOT does not start with digraph")

    if dot_text.count("{") != dot_text.count("}"):
        raise ValueError("Unbalanced braces")

    return dot_text


🔥 Updated rendering/dot_sanitizer.py
import re
import html


def clean_and_validate_dot(dot_text: str):

    # Remove markdown fences
    dot_text = re.sub(r"```.*?```", "", dot_text, flags=re.DOTALL)

    # Unescape HTML entities
    dot_text = html.unescape(dot_text)

    # Extract DOT block starting at digraph
    match = re.search(r"(digraph\s+[^{]+\{.*\})", dot_text, re.DOTALL)

    if not match:
        raise ValueError(
            "No valid digraph block found in LLM output.\n\nRAW OUTPUT:\n"
            + dot_text[:500]
        )

    dot_block = match.group(1).strip()

    # Validate braces balance
    if dot_block.count("{") != dot_block.count("}"):
        raise ValueError("Unbalanced braces in DOT.")

    return dot_block

Sure, here you go:

It won’t break.

🔥 If You Want Bulletproof Mode

Add this to your generate_dot():

stop=["```", "Here is", "Sure"]

But sanitizer fix is enough.

🧠 Extra Debug Tip

Temporarily add this inside agent.py before sanitizer:

print("\n=== RAW LLM OUTPUT ===\n")
print(dot_code)
print("\n======================\n")

In [ ]:
✅ Step 1 — Add SQL Chunker

Create new file:

parsing/sql_chunker.py
import re


def chunk_sql(sql_text, max_chars=4000):

    if len(sql_text) <= max_chars:
        return [sql_text]

    # Split by LEFT JOIN boundaries
    chunks = re.split(r"\bLEFT\s+JOIN\b", sql_text, flags=re.IGNORECASE)

    rebuilt_chunks = []

    current = ""

    for part in chunks:
        candidate = current + " LEFT JOIN " + part if current else part

        if len(candidate) > max_chars:
            rebuilt_chunks.append(current.strip())
            current = part
        else:
            current = candidate

    if current:
        rebuilt_chunks.append(current.strip())

    return rebuilt_chunks
✅ Step 2 — Modify llm_normalizer.py

Import chunker:

from parsing.sql_chunker import chunk_sql

Modify normalize_stage():

def normalize_stage(stage_id, raw):

    sql_chunks = chunk_sql(raw["sql"])

    combined = {
        "stage_id": stage_id,
        "primary_tables": [],
        "joins": [],
        "filters": [],
        "transformations": []
    }

    for chunk in sql_chunks:

        response = client.chat.completions.create(
            model=AZURE_OPENAI_DEPLOYMENT,
            temperature=0,
            messages=[
                {"role": "system", "content": "You extract SQL lineage strictly."},
                {
                    "role": "user",
                    "content": PROMPT +
                    f"\nStage ID: {stage_id}\n\nSQL:\n{chunk}\n\nTRANSFORM:\n{raw['transform']}"
                }
            ]
        )

        content = response.choices[0].message.content.strip()

        try:
            parsed = json.loads(content)

            combined["primary_tables"].extend(parsed.get("primary_tables", []))
            combined["joins"].extend(parsed.get("joins", []))
            combined["filters"].extend(parsed.get("filters", []))
            combined["transformations"].extend(parsed.get("transformations", []))

        except:
            continue

    # Deduplicate
    combined["primary_tables"] = list(set(combined["primary_tables"]))

    return combined


    import re
import html


def clean_and_validate_dot(dot_text: str):

    if not dot_text:
        raise ValueError("LLM returned empty response.")

    # 1️⃣ Remove markdown fences like ```dot ``` or ```
    dot_text = re.sub(r"```[\w]*", "", dot_text)
    dot_text = re.sub(r"```", "", dot_text)

    # 2️⃣ Remove triple single quotes if present
    dot_text = dot_text.replace("'''", "")

    # 3️⃣ Remove leading/trailing whitespace
    dot_text = dot_text.strip()

    # 4️⃣ Unescape HTML entities
    dot_text = html.unescape(dot_text)

    # 5️⃣ Extract only digraph block
    match = re.search(r"(digraph\s+[^{]+\{.*\})", dot_text, re.DOTALL)

    if not match:
        raise ValueError(
            "No valid digraph block found.\n\nRAW OUTPUT:\n"
            + dot_text[:800]
        )

    dot_block = match.group(1).strip()

    # 6️⃣ Validate braces
    if dot_block.count("{") != dot_block.count("}"):
        raise ValueError("Unbalanced braces in DOT.")

    return dot_block

In [ ]:
PROMPT = """
You are a professional Graphviz workflow compiler.

Generate COMPLETE valid Graphviz DOT code.

STRICT REQUIREMENTS:

- rankdir=LR
- bgcolor="white"
- node shape=plain
- fontname="Helvetica"
- Use subgraph cluster_workflow
- Do NOT use markdown
- Do NOT use ```
- Do NOT add explanations

NODE TEMPLATE (MUST FOLLOW EXACTLY):

STAGE_ID [
    label=<
        <table border="0" cellborder="1" cellspacing="0" cellpadding="6">
            <tr>
                <td bgcolor="#E3F2FD">
                    <b>STAGE_ID</b>
                </td>
            </tr>
            <tr>
                <td align="left">
                    <b>Inputs:</b> comma separated inputs or None<br/>
                    <b>Outputs:</b> comma separated outputs or None
                </td>
            </tr>
        </table>
    >
];

Create edges using inputs → outputs mapping.

Output must start with:

digraph WORKFLOW_NAME {
"""